In [2]:
!pip install newspaper4k feedparser pandas requests lxml

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.8 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/3.8 MB 621.2 kB/s eta 0:00:06
   ----- ---------------------------------- 0.5/3.8 MB 621.2 kB/s eta 0:00:06
   -------- ------------------------------- 0.8/3.8 MB 633.2 kB/s eta 0:00:05
   ----------- -------


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import feedparser
import pandas as pd
from newspaper import Article
import time
import requests

# RSS Feed yang valid dan teruji (2024-2025)
RSS_FEEDS = {
    "Kompas": [
        "https://www.kompas.com/rss/feed.xml",
    ],
    "Detik": [
        "https://rss.detik.com/index.php/detikcom",
        "https://feed.detik.com/detikcom",
    ],
    "CNN Indonesia": [
        "https://www.cnnindonesia.com/rss",
    ],
    "Republika": [
        "https://www.republika.co.id/rss",
    ],
    "Antara": [
        "https://www.antaranews.com/rss/terkini.xml",
    ],
}

HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
}

def parse_rss_safe(url):
    """Parse RSS dengan requests dulu baru feedparser — hindari RemoteDisconnected"""
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)
        resp.raise_for_status()
        feed = feedparser.parse(resp.text)
        return feed
    except Exception as e:
        print(f"  ⚠ RSS gagal ({url}): {e}")
        return None

def scrape_article(url):
    """Ambil konten artikel"""
    try:
        article = Article(url, language='id', request_timeout=10)
        article.download()
        article.parse()
        if not article.text or len(article.text) < 150:
            return None
        return {
            "judul": article.title or "",
            "isi": article.text,
            "tanggal": str(article.publish_date) if article.publish_date else "",
            "penulis": ", ".join(article.authors) if article.authors else "",
        }
    except Exception as e:
        return None

def scrape_from_rss(feed_url, sumber, max_artikel=80):
    feed = parse_rss_safe(feed_url)
    if not feed or not feed.entries:
        print(f"  ✗ Tidak ada entry di {feed_url}")
        return []

    print(f"  → {len(feed.entries)} entry ditemukan")
    hasil = []

    for i, entry in enumerate(feed.entries[:max_artikel]):
        url = entry.get("link", "")
        judul_preview = entry.get("title", "")[:55]
        print(f"  [{i+1:02d}] {judul_preview}...")

        if not url:
            continue

        # Coba ambil konten artikel
        data = scrape_article(url)

        if data:
            # Ambil kategori dari tag RSS jika ada
            tags = entry.get("tags", [])
            kategori = tags[0].get("term", "umum") if tags else "umum"

            data.update({
                "url": url,
                "sumber": sumber,
                "kategori": kategori,
            })
            hasil.append(data)
            print(f"       ✓ {len(data['isi'])} karakter")
        else:
            print(f"       ✗ skip (terlalu pendek / error)")

        time.sleep(1.5)  # rate limiting

    return hasil


# ── MAIN ──────────────────────────────────────────────────────────────
semua_artikel = []

for sumber, feeds in RSS_FEEDS.items():
    print(f"\n{'='*50}")
    print(f"[{sumber}]")
    for feed_url in feeds:
        artikel = scrape_from_rss(feed_url, sumber, max_artikel=80)
        if artikel:
            semua_artikel.extend(artikel)
            print(f"  ✅ {len(artikel)} artikel berhasil dari {sumber}")
            break  # kalau sudah berhasil, skip URL fallback berikutnya
        else:
            print(f"  Mencoba URL berikutnya...")

# ── SIMPAN CSV ─────────────────────────────────────────────────────────
df = pd.DataFrame(semua_artikel)
print(f"\nTotal sebelum cleaning: {len(df)}")

df = df.dropna(subset=["isi", "judul"])
df = df[df["isi"].str.len() > 200]
df = df.drop_duplicates(subset=["judul"])
df = df.reset_index(drop=True)

df.to_csv("dataset_berita.csv", index=False, encoding="utf-8-sig")
print(f"✅ Disimpan: {len(df)} artikel → dataset_berita.csv")
print(df[["sumber", "judul", "kategori"]].groupby("sumber").count())


[Kompas]
  ⚠ RSS gagal (https://www.kompas.com/rss/feed.xml): 404 Client Error: Not Found for url: https://www.kompas.com/rss/feed.xml
  ✗ Tidak ada entry di https://www.kompas.com/rss/feed.xml
  Mencoba URL berikutnya...

[Detik]
  ⚠ RSS gagal (https://rss.detik.com/index.php/detikcom): ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
  ✗ Tidak ada entry di https://rss.detik.com/index.php/detikcom
  Mencoba URL berikutnya...
  ⚠ RSS gagal (https://feed.detik.com/detikcom): HTTPSConnectionPool(host='feed.detik.com', port=443): Max retries exceeded with url: /detikcom (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x00000159968BADD0>: Failed to resolve 'feed.detik.com' ([Errno 11001] getaddrinfo failed)"))
  ✗ Tidak ada entry di https://feed.detik.com/detikcom
  Mencoba URL berikutnya...

[CNN Indonesia]
  → 100 entry ditemukan
  [01] Veda Ega Bakal Rasakan Q1 untuk Kali Pertama di Moto3 S...
       ✓ 1432 karak

In [5]:
import feedparser
import pandas as pd
import requests
from newspaper import Article
import time

RSS_FEEDS = [
    ("Antara",      "https://www.antaranews.com/rss/terkini.xml"),
    ("Antara Ekonomi", "https://www.antaranews.com/rss/ekonomi.xml"),
    ("Antara Olahraga", "https://www.antaranews.com/rss/olahraga.xml"),
    ("Kompas",      "https://www.kompas.com/rss/feed.xml"),
    ("Republika",   "https://www.republika.co.id/rss"),
    ("CNN Indonesia","https://www.cnnindonesia.com/rss"),
    ("Detik",       "https://feed.detik.com/detikcom"),
]

HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"}
TARGET = 1000

def get_urls_from_rss(feed_url, max_per_feed=200):
    try:
        resp = requests.get(feed_url, headers=HEADERS, timeout=15)
        feed = feedparser.parse(resp.text)
        return [e.get("link", "") for e in feed.entries[:max_per_feed] if e.get("link")]
    except:
        return []

def scrape_isi(url):
    try:
        article = Article(url, language='id', request_timeout=10)
        article.download()
        article.parse()
        if article.text and len(article.text) > 200:
            return article.text.strip()
    except:
        pass
    return None

# ── Kumpulkan semua URL dulu ──
print("📡 Mengumpulkan URL dari RSS...")
semua_url = []
for nama, feed_url in RSS_FEEDS:
    urls = get_urls_from_rss(feed_url)
    semua_url.extend(urls)
    print(f"  {nama}: {len(urls)} URL")

# Hapus duplikat
semua_url = list(dict.fromkeys(semua_url))
print(f"\nTotal URL unik: {len(semua_url)}")

# ── Scrape sampai 1000 artikel ──
print(f"\n🔍 Scraping isi berita (target: {TARGET})...")
hasil = []
gagal = 0

for i, url in enumerate(semua_url):
    if len(hasil) >= TARGET:
        break

    isi = scrape_isi(url)
    status = "✓" if isi else "✗"
    if isi:
        hasil.append({"isi": isi})
    else:
        gagal += 1

    print(f"  [{len(hasil):04d}/{TARGET}] {status} {url[:60]}")
    time.sleep(0.8)

# ── Simpan CSV ──
df = pd.DataFrame(hasil)
df = df.drop_duplicates(subset=["isi"])
df = df.reset_index(drop=True)

df.to_csv("dataset_berita.csv", index=False, encoding="utf-8-sig")
print(f"\n✅ Selesai: {len(df)} artikel → dataset_berita.csv")
print(f"   Gagal/skip: {gagal}")

📡 Mengumpulkan URL dari RSS...
  Antara: 50 URL
  Antara Ekonomi: 20 URL
  Antara Olahraga: 20 URL
  Kompas: 0 URL
  Republika: 15 URL
  CNN Indonesia: 100 URL
  Detik: 0 URL

Total URL unik: 205

🔍 Scraping isi berita (target: 1000)...
  [0001/1000] ✓ https://www.antaranews.com/berita/5541659/revitalisasi-cagar
  [0002/1000] ✓ https://www.antaranews.com/berita/5541656/suyatno-ditunjuk-s
  [0003/1000] ✓ https://www.antaranews.com/berita/5541653/pemkot-kediri-inve
  [0004/1000] ✓ https://www.antaranews.com/berita/5541652/pacific-caesar-pak
  [0004/1000] ✗ https://www.antaranews.com/video/5541569/festival-sagu-papua
  [0005/1000] ✓ https://www.antaranews.com/video/5541584/kampung-mrican-slem
  [0006/1000] ✓ https://www.antaranews.com/video/5541635/singgung-konflik-ti
  [0007/1000] ✓ https://www.antaranews.com/berita/5541648/kpk-periksa-27-pej
  [0008/1000] ✓ https://www.antaranews.com/berita/5541644/pbb-sampaikan-bela
  [0009/1000] ✓ https://www.antaranews.com/berita/5541640/pg-ngadirejo